# 02 · FastText Implementation

In [2]:
import sys
sys.path.append('..')

import os
import re
import gzip
import shutil

import numpy as np
import pandas as pd
import torch
import urllib.request

from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# https://radimrehurek.com/gensim/models/fasttext.html
from gensim.models.fasttext import load_facebook_model

print(f'device : {"cuda" if torch.cuda.is_available() else "cpu"}')

/home/hice1/rdurai6/scratch/project/graph-semantic-research-main/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device : cuda


In [3]:
input_dir  = '../data/'
output_dir = os.path.expanduser('../outputs/')
model_dir  = os.path.expanduser('../models/')
os.makedirs(output_dir, exist_ok=True)
os.makedirs(model_dir,  exist_ok=True)

FASTTEXT_URL = 'https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz'
FASTTEXT_BIN = os.path.join(model_dir, 'cc.en.300.bin')

save_path = os.path.join(output_dir, 'title_embeddings_fasttext.npy')

In [4]:
if os.path.exists(FASTTEXT_BIN):
    print(f'Model already present: {FASTTEXT_BIN}')
else:
    gz_path = FASTTEXT_BIN + '.gz'
    if not os.path.exists(gz_path):
        print(f'Downloading {FASTTEXT_URL}')
        urllib.request.urlretrieve(FASTTEXT_URL, gz_path)
    print('Decompressing')
    with gzip.open(gz_path, 'rb') as f_in, open(FASTTEXT_BIN, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(gz_path)

print('Model downloaded successfully!')

Model already present: ../models/cc.en.300.bin
Model downloaded successfully!


In [5]:
df = pd.read_parquet(input_dir + 'amazon_products.parquet')
print('Data loaded successfully!')

Data loaded successfully!


In [6]:
from scripts.embedding_pipeline_st import EmbeddingPipeline
from typing import Optional

REGEX = re.compile(r'[^a-z0-9\s]')

def tokenise(text):
    if not isinstance(text, str) or not text.strip():
        return []
    return REGEX.sub(' ', text.lower()).split()


class FastTextEmbeddingPipeline(EmbeddingPipeline):
    """
    Subclass of EmbeddingPipeline that replaces the SentenceTransformer backend with a gensim fastText model.

    model_name: path to a Facebook fastText .bin file
    l2_normalise: L2-normalise each embedding (default True)
    """

    def __init__(self, model_name: str, l2_normalise: bool = True):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model_name = model_name
        self.l2_normalise = l2_normalise
        self.model = load_facebook_model(model_name)
        self._dim  = self.model.vector_size
        print(f'Model loaded')
        print(f'vocab={len(self.model.wv.key_to_index):,}')
        print(f'dim={self._dim}')

    def embed_sentences(self, sentences: list, batch_size: int) -> np.ndarray:
        N = len(sentences)
        all_embeds = np.zeros((N, self._dim), dtype=np.float32)

        for start in tqdm(range(0, N, batch_size), unit='batch'):
            end = min(start + batch_size, N)
            for i, text in enumerate(sentences[start:end]):
                tokens = tokenise(text)
                if not tokens:
                    continue

                vectors = np.stack([self.model.wv[tok] for tok in tokens])
                all_embeds[start+i] = vectors.mean(axis=0)

        if self.l2_normalise:
            t = torch.from_numpy(all_embeds)
            all_embeds = torch.nn.functional.normalize(t, p=2, dim=1).numpy()

        return all_embeds

    def compute_embeddings(self, df: pd.DataFrame, column: str, batch_size: int = 128, save_path: Optional[str] = None) -> np.ndarray:
        if column not in df.columns:
            raise ValueError(f"Column '{column}' not found in DataFrame.")

        sentences = df[column].fillna('').astype(str).tolist()

        print(f'Starting embedding process for {len(sentences)} items')
        embeddings = self.embed_sentences(sentences, batch_size)

        print(f'Computation complete. Embedding shape: {embeddings.shape}')

        if save_path:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            np.save(save_path, embeddings)
            print(f'Embeddings saved to: {save_path}')

        return embeddings

In [7]:
pipeline = FastTextEmbeddingPipeline(model_name=FASTTEXT_BIN)
embeddings = pipeline.compute_embeddings(df=df, column='title', batch_size=128, save_path=save_path)

Model loaded
vocab=2,000,000
dim=300
Starting embedding process for 548552 items


100%|██████████| 4286/4286 [00:27<00:00, 157.59batch/s]


Computation complete. Embedding shape: (548552, 300)
Embeddings saved to: ../outputs/title_embeddings_fasttext.npy


In [8]:
norms = np.linalg.norm(embeddings, axis=1)

print(f'Embedding dim : {embeddings.shape[1]}')
print(f'Total nodes : {embeddings.shape[0]:,}')
print(f'Zero-vector rows (titles with no recognisable tokens) : {(norms < 1e-8).sum():,}')
print(f'Norm mean / std : {norms.mean():.4f} / {norms.std():.4f}')

Embedding dim : 300
Total nodes : 548,552
Zero-vector rows (titles with no recognisable tokens) : 5,875
Norm mean / std : 0.9893 / 0.1029


In [9]:
def test_embeddings(df, embeddings, query_index, top_k=5):
    query_title = df.iloc[query_index]['title']
    query_vec = embeddings[query_index].reshape(1, -1)

    print(f'--- QUERY PRODUCT (Index {query_index}) ---')
    print(f'Title: {query_title}\n')

    scores  = cosine_similarity(query_vec, embeddings).flatten()
    indices = np.argsort(scores)[-(top_k + 1):][::-1]

    print(f'--- TOP {top_k} SIMILAR PRODUCTS ---')
    for i in indices:
        if i == query_index:
            continue

        similarity  = scores[i]
        match_title = df.iloc[i]['title']
        print(f'[{similarity:.4f}] {match_title}')

In [10]:
test_index = 100
test_embeddings(df, embeddings, query_index=test_index)

--- QUERY PRODUCT (Index 100) ---
Title: Guide to Effective Staff Development in Health Care Organizations : A Systems Approach to Successful Training (J-B AHA Press)



--- TOP 5 SIMILAR PRODUCTS ---
[0.9425] Disease Management : A Systems Approach to Improving Patient Outcomes (J-B AHA Press)
[0.9349] Evolving Practices in Human Resource Management : Responses to a Changing World of Work (J-B SIOP Professional Practice Series)
[0.9234] Insights in Decision Making : A Tribute to Hillel J. Einhorn
[0.9162] Spelling Smart! : A Ready-to-Use Activities Program for Students with Spelling Difficulties (J-B Ed: Ready-to-Use Activities)
[0.9147] Smart Alliances : A Practical Guide to Repeatable Success (J-B BAH Strategy & Business Series)


## Read the Parquet File and load to a Dataframe

In [14]:
# pip install fastparquet
import pandas as pd

file_path = r"../data/amazon_data.parquet"
df = pd.read_parquet(file_path)
print(df.head())

print("Data loaded successfully!")

print('Number of Products (Rows):', len(df))

  Id Active        ASIN                                              Title  \
0  0      N  0771044445                                               None   
1  1      Y  0827229534            Patterns of Preaching: A Sermon Sampler   
2  2      Y  0738700797                         Candlemas: Feast of Flames   
3  3      Y  0486287785   World War II Allied Fighter Planes Trading Cards   
4  4      Y  0842328327  Life Application Bible Commentary: 1 and 2 Tim...   

  Group Salesrank similarCount  \
0  None      None         None   
1  Book    396585            5   
2  Book    168596            5   
3  Book   1270652            0   
4  Book    631289            5   

                                        SimilarASINs  \
0                                               None   
1  b'["0804215715","156101074X","0687023955","068...   
2  b'["0738700827","1567184960","1567182836","073...   
3                                              b'[]'   
4  b'["0842328130","0830818138","0842330313","

## Reduce to 5 core (Each nodes must have k neighbors: k = 5)

In [18]:
import networkx as nx
import json

# Decode bytes -> string -> parse JSON list, handling None/empty
def parse_similar(val):
    if val is None:
        return []
    if isinstance(val, (bytes, bytearray)):
        val = val.decode('utf-8')
    if isinstance(val, str):
        try:
            return json.loads(val)
        except (json.JSONDecodeError, ValueError):
            return []
    if isinstance(val, list):
        return val
    return []

# Create all unique edges (Undirected)
edges = set()
for idx, row in df.iterrows():
    u = row['ASIN']
    for v in parse_similar(row['SimilarASINs']):
        # Sort to ensure (A, B) and (B, A) are treated as the same undirected edge
        edge = tuple(sorted((u, v)))
        edges.add(edge)

# Build the graph
G = nx.Graph()
G.add_edges_from(edges)

print(f'Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.')

Graph has 554789 nodes and 1545228 edges.


In [19]:
# Extract the 5-core (iterative prune until all nodes have degree >= 5)
G_FiveCore = nx.k_core(G, k=5)

# Extract the Largest Connected Component
lcc_nodes = max(nx.connected_components(G_FiveCore), key=len)
G_final = G_FiveCore.subgraph(lcc_nodes).copy()

print(f'Nodes in Final Graph: {G_final.number_of_nodes()}')

Nodes in Final Graph: 111624


In [20]:
df_lcc = df[df['ASIN'].isin(G_final.nodes())].copy()
print(df_lcc['Group'].value_counts())

Group
Book     78311
Music     9890
DVD       9019
Video     7008
Name: count, dtype: int64


In [21]:
# Identify valid ASINs (Active 'Y' AND has a valid Title AND has a rating)
valid_nodes_df = df_lcc[
    (df_lcc['Active'] == 'Y') &
    (df_lcc['Title'].notna()) &
    (df_lcc['Title'] != 'None') &
    (df_lcc['avg_rating'] > 0)
].copy()

valid_asins = set(valid_nodes_df['ASIN'])

# Create a subgraph containing ONLY these valid nodes
G_valid = G_final.subgraph(valid_asins).copy()

# Re-run 5-Core Pruning
# Some nodes will drop out because their neighbors were removed in step 1
G_FiveCore_Final = nx.k_core(G_valid, k=5)

# Extract the final Largest Connected Component (LCC)
lcc_nodes_final = max(nx.connected_components(G_FiveCore_Final), key=len)
G_final_clean = G_FiveCore_Final.subgraph(lcc_nodes_final).copy()

print(f'Final Research Nodes: {G_final_clean.number_of_nodes()}')

Final Research Nodes: 24492


In [22]:
final_asins = list(G_final_clean.nodes())
df_active = df[df['ASIN'].isin(final_asins)].copy()
print(f'Columns in df_active: {df_active.columns.tolist()}')

Columns in df_active: ['Id', 'Active', 'ASIN', 'Title', 'Group', 'Salesrank', 'similarCount', 'SimilarASINs', 'Categories', 'CleanCategories', 'reviews_count', 'avg_rating']


# Unique Categories

In [23]:
all_paths = df_active['CleanCategories'].explode().dropna().unique()
print(f'Total Unique Categories: {len(all_paths)}')

category_to_id = {name: i for i, name in enumerate(all_paths)}

def map_to_ids(cat_list):
    if isinstance(cat_list, list):
        return list(set(category_to_id[cat] for cat in cat_list if cat in category_to_id))
    return []

df_active['CategoryIDs'] = df_active['CleanCategories'].apply(map_to_ids)
print(df_active[['CleanCategories', 'CategoryIDs']].head())

Total Unique Categories: 14869
                                      CleanCategories CategoryIDs
2   b'["Books > Subjects > Religion & Spirituality...          []
10  b'["Books > Subjects > Literature & Fiction > ...          []
21  b'[" > DVD > Genres > Drama"," > DVD > Genres ...          []
43  b'["Books > Subjects > Literature & Fiction > ...          []
82  b'["Books > Subjects > Literature & Fiction > ...          []


In [24]:
rank_min = df_active['Salesrank'].min()
rank_max = df_active['Salesrank'].max()
print(f'Salesrank Range: {rank_min} to {rank_max}')
print(df_active['Salesrank'].describe())

Salesrank Range: 0 to 99997
count     24492
unique    22778
top        5926
freq          6
Name: Salesrank, dtype: object


# Add New Id field

In [25]:
df_active['New_Id'] = range(len(df_active))

cols = ['New_Id'] + [c for c in df_active.columns if c != 'New_Id']
df_active = df_active[cols]
print(df_active[['New_Id', 'Id', 'ASIN', 'CategoryIDs']].head())

df_final = df_active[['New_Id', 'Title', 'Group', 'Salesrank', 'reviews_count', 'CategoryIDs']].copy()
df_final['Salesrank'] = pd.to_numeric(df_final['Salesrank'], errors='coerce').fillna(0).astype(int)
df_final['reviews_count'] = df_final['reviews_count'].astype(int)
print(df_final.head())

    New_Id  Id        ASIN CategoryIDs
2        0   2  0738700797          []
10       1  10  0375709363          []
21       2  21  0790747324          []
43       3  43  0486268780          []
82       4  82  0140430407          []
    New_Id                                              Title Group  \
2        0                         Candlemas: Feast of Flames  Book   
10       1                             The Edward Said Reader  Book   
21       2                                   The Time Machine   DVD   
43       3             Selected Poems (Dover Thrift Editions)  Book   
82       4  Puddnhead Wilson : And, Those Extraordinary Tw...  Book   

    Salesrank  reviews_count CategoryIDs  
2      168596             12          []  
10     220379              6          []  
21        795            140          []  
43     260300              2          []  
82      60668              4          []  


# Encode Title

## FastText

In [26]:
# df_active rows are a subset of the original df — use the original positional index
# to slice the correct rows out of the full embeddings array.
active_indices = df_active.index.tolist()
embeddings_active = embeddings[active_indices]  # shape: (N_active, 300)

print(f'Embeddings subset shape: {embeddings_active.shape}')


df_final['Title_Embeddings'] = list(embeddings_active)
print(df_final[['Title', 'Title_Embeddings']].head())
print(df_final['Title_Embeddings'].size, df_final['Title_Embeddings'].iloc[0].shape)

Embeddings subset shape: (24492, 300)
                                                Title  \
2                          Candlemas: Feast of Flames   
10                             The Edward Said Reader   
21                                   The Time Machine   
43             Selected Poems (Dover Thrift Editions)   
82  Puddnhead Wilson : And, Those Extraordinary Tw...   

                                     Title_Embeddings  
2   [-0.029913299, -0.03930758, 0.013446038, -0.05...  
10  [-0.034466524, 0.041230578, 0.029573848, 0.050...  
21  [-0.024597336, 0.07599726, -0.043120153, 0.086...  
43  [-0.023997543, 0.09863641, -0.07646583, 0.0772...  
82  [-0.036582638, 0.027652076, -0.027985355, 0.09...  
24492 (300,)


In [27]:
# Extract embeddings to CSV — used as node features for the GNN
node_features_path = input_dir + 'node_features.csv'
embeddings_df = pd.DataFrame(df_final['Title_Embeddings'].tolist())
embeddings_df.to_csv(node_features_path, index=False, header=False)

size_mb = os.path.getsize(node_features_path) / (1024 * 1024)
print(f'node_features.csv saved — {size_mb:.2f} MB')

node_features.csv saved — 86.04 MB


# Generate the Edge Index

In [29]:
asin_to_id = pd.Series(df_active.New_Id.values, index=df_active.ASIN).to_dict()

# Parse SimilarASINs bytes → list before exploding
df_active['SimilarASINs_parsed'] = df_active['SimilarASINs'].apply(parse_similar)

edges_df = df_active[['New_Id', 'SimilarASINs_parsed']].explode('SimilarASINs_parsed')
edges_df = edges_df.rename(columns={'SimilarASINs_parsed': 'SimilarASINs'})
edges_df['Target_Id'] = edges_df['SimilarASINs'].map(asin_to_id)
edge_index_df = edges_df.dropna(subset=['Target_Id']).copy()
edge_index_df['Target_Id'] = edge_index_df['Target_Id'].astype(int)
edge_list = edge_index_df[['New_Id', 'Target_Id']].rename(
    columns={'New_Id': 'source', 'Target_Id': 'target'}
)
print(f'Total edges: {len(edge_list)}')
print(edge_list.head())

edge_list_path = input_dir + 'edge_list.txt'
edge_list.to_csv(edge_list_path, sep=' ', index=False, header=False)
print(f'Edge list saved to {edge_list_path}')

Total edges: 113276
   source  target
2       0   19591
2       0    2786
2       0   17782
2       0    7551
2       0   13262
Edge list saved to ../data/edge_list.txt


## Ratings

In [30]:
ratings_only = df_active['avg_rating'].round().astype(int)
labels_0_indexed = ratings_only - 1

labels_path = input_dir + 'labels.csv'
labels_0_indexed.to_csv(labels_path, index=False, header=False)
print(f'Unique labels saved: {labels_0_indexed.unique()}')

Unique labels saved: [3 4 2 1 0]


# Generate NPZ

In [32]:
from scripts.create_npz import create_npx

npz_path = output_dir + 'amazon_fasttext.npz'
create_npx.create_npz_dataset(
    node_features_file=input_dir + 'node_features.csv',
    labels_file=input_dir + 'labels.csv',
    edges_file=input_dir + 'edge_list.txt',
    num_splits=10,
    output_file_name=npz_path
)

size_mb = os.path.getsize(npz_path) / (1024 * 1024)
print(f'NPZ File Size: {size_mb:.2f} MB')

Verified Split 0 - Train: 14695, Val: 4898, Test: 4899
no of nodes 24492
no of features per node 300
no of node labels, should matach no of nodes 24492
no of edges 113276
NPZ File Size: 30.65 MB


## Validation

In [33]:
data = np.load(npz_path)
print('Keys in NPZ:', list(data.keys()))
for key in data.keys():
    print(f'{key} shape: {data[key].shape}')

labels = data['node_labels']
unique, counts = np.unique(labels, return_counts=True)
print(dict(zip(unique, counts)))

Keys in NPZ: ['node_features', 'node_labels', 'edges', 'train_masks', 'val_masks', 'test_masks']
node_features shape: (24492, 300)
node_labels shape: (24492,)
edges shape: (113276, 2)
train_masks shape: (10, 24492)
val_masks shape: (10, 24492)
test_masks shape: (10, 24492)
{0: 17, 1: 272, 2: 772, 3: 16871, 4: 6560}
